# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: fcab2fd7-9e8d-4b6b-b89d-60727761fb47
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session fcab2fd7-9e8d-4b6b-b89d-60727761fb47 to get into ready status...
Session fcab2fd7-9e8d-4b6b-b89d-60727761fb47 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
products_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='products')
category_translations_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='product_category_name_translation')

products_dyf.printSchema()
category_translations_dyf.printSchema()

root
|-- product_id: string
|-- product_category_name: string
|-- product_name_lenght: long
|-- product_description_lenght: long
|-- product_photos_qty: long
|-- product_weight_g: long
|-- product_length_cm: long
|-- product_height_cm: long
|-- product_width_cm: long

root
|-- col0: string
|-- col1: string


In [3]:
# fix the typos in product_name_length and product_description_length columns
products_dyf = products_dyf.apply_mapping([
    ("product_id","string","product_id","string"),
    ("product_category_name","string","product_category_name","string"),
    ("product_name_lenght","bigint","product_name_length","bigint"),
    ("product_description_lenght","bigint","product_description_length","bigint"),
    ("product_photos_qty","bigint","product_photos_qty","bigint"),
    ("product_weight_g","bigint","product_weight_g","bigint"),
    ("product_length_cm","bigint","product_length_cm","bigint"),
    ("product_height_cm","bigint","product_height_cm","bigint"),
    ("product_width_cm","bigint","product_width_cm","bigint"),
])

# wrong crawler schema, fixed this
category_translations_dyf = category_translations_dyf.apply_mapping([
    ("col0", "string", "product_category_name", "string"),
    ("col1", "string", "product_category_name_english", "string")
])

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [4]:
# I transform both DynamicFrames into pyspark Dataframes to use pyspark SQl
products_df = products_dyf.toDF()
category_translations_df = category_translations_dyf.toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
products_df.createOrReplaceTempView("products")
category_translations_df.createOrReplaceTempView("category_translations")

joined_product_df = spark.sql("""
    SELECT * FROM products p1
    JOIN category_translations c1 
    ON p1.product_category_name=c1.product_category_name
""")

joined_product_df = joined_product_df.limit(1000)

In [6]:
# Step 1: Drop all columns whose 'product_id' column is null
cleaned_jp_df = joined_product_df.dropna(subset=["product_id"])

# Step 2: Calculate the dimensions of the product
cleaned_jp_df = cleaned_jp_df.withColumn("product_dimension_cm3", cleaned_jp_df["product_length_cm"] * 
                                                                  cleaned_jp_df["product_height_cm"] *
                                                                  cleaned_jp_df["product_width_cm"])

In [7]:
# Convert the pyspark Dataframe back into a DynamicFrame
from awsglue.dynamicframe import DynamicFrame

cleaned_dyf = DynamicFrame.fromDF(cleaned_jp_df, glueContext, "products_dyf")

In [8]:
# Store the cleaned dataset as Parquet in S3
s3output = glueContext.getSink(
  path="s3://bucket181rt2/clean/products",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_clean", catalogTableName="products"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(cleaned_dyf)